In [1]:
# import statements --> SOME OF THESE MAY NOT BE NEEDED
%matplotlib inline 
import numpy as np 

from matplotlib import pyplot as plt 
from matplotlib.cm import get_cmap

from sklearn import neighbors


import umap
import umap.plot
from umap.umap_ import nearest_neighbors
import pandas as pd

import plotly.express as px


import pickle

pd.set_option('display.max_columns', None)

#BRETT NOTEBOOK IMPORTS
from pathlib import Path

from importlib import reload
import utils
reload(utils)
from minisom import MiniSom


# imports from animation_simplified2 notebook
import seaborn as sns
sns.set(style='white', rc={'figure.figsize':(14, 12), 'animation.html': 'html5'})
import os
import copy

/Applications/anaconda3/envs/env1/lib/python3.11/site-packages/umap/plot.py:203: NumbaDeprecationWarning: The keyword argument 'nopython=False' was supplied. From Numba 0.59.0 the default is being changed to True and use of 'nopython=False' will raise a warning as the argument will have no effect. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  @numba.jit(nopython=False)


In [2]:
import umapz2

In [3]:
data = pd.read_parquet('df7_maps_250819.parquet')
spec95 = pd.read_parquet('spec95_ugrizyJH_maps_250819.parquet')
data = data.drop(columns=['SOM-1', 'SOM-2', 'UMAP3D-1', 'UMAP3D-2', 'UMAP3D-3'])
spec95 = spec95.drop(columns=['SOM-1', 'SOM-2', 'UMAP3D-1', 'UMAP3D-2', 'UMAP3D-3'])
data = data[~data['ID'].isin(spec95[spec95['specz']<0.01]['ID'])]
spec95 = spec95[spec95['specz']>0.01]
spec95

,ID,RA,DEC,FLAG_COMBINED,CFHT_u_FLUX,CFHT_u_FLUXERR,CFHT_u_MAGERR,HSC_g_FLUX,HSC_g_FLUXERR,HSC_g_MAGERR,HSC_r_FLUX,HSC_r_FLUXERR,HSC_r_MAGERR,HSC_i_FLUX,HSC_i_FLUXERR,HSC_i_MAGERR,HSC_z_FLUX,HSC_z_FLUXERR,HSC_z_MAGERR,HSC_y_FLUX,HSC_y_FLUXERR,HSC_y_MAGERR,UVISTA_J_FLUX,UVISTA_J_FLUXERR,UVISTA_J_MAGERR,UVISTA_H_FLUX,UVISTA_H_FLUXERR,UVISTA_H_MAGERR,UVISTA_Ks_FLUX,UVISTA_Ks_FLUXERR,UVISTA_Ks_MAGERR,IRAC_CH1_FLUX,IRAC_CH1_FLUXERR,IRAC_CH1_MAGERR,IRAC_CH2_FLUX,IRAC_CH2_FLUXERR,IRAC_CH2_MAGERR,lp_photoz,lp_type,lp_zp_2,lp_NbFilt,lp_zq,lp_model,lp_age,lp_dust,lp_Attenuation,lp_MNUV,lp_MR,lp_MJ,lp_mass_med,lp_mass_med_min68,lp_mass_med_max68,lp_mass_best,lp_SFR_med,lp_sSFR_best,CFHT_u_MAG,HSC_g_MAG,HSC_r_MAG,HSC_i_MAG,HSC_z_MAG,HSC_y_MAG,UVISTA_J_MAG,UVISTA_H_MAG,UVISTA_Ks_MAG,IRAC_CH1_MAG,IRAC_CH2_MAG,u-g,g-r,r-i,i-z,z-y,y-J,J-H,nearest_cat1_ID,nearest_cat1_sep,Id_specz,Id_original,ra_original,dec_original,ra_corrected,dec_corrected,Priority,specz,flag,Confidence_level,survey,compilation_year,public_or_private,Id_COS20_Classic,ra_COS20_Classic,dec_COS20_Classic,Id_COS20_Farmer,ra_COS20_Farmer,dec_COS20_Farmer,Id_COSMOS15,ra_COSMOS15,dec_COSMOS15,Id_COSMOS09,ra_COSMOS09,dec_COSMOS09,photoz,photoz_type
0,535,150.726526,1.795723,0,3.135322,0.004911,0.001979,3.543392,0.005600,0.001707,4.149280,0.006839,0.001707,5.531073,0.006686,0.001311,9.094858,0.011055,0.001310,6.137582,0.023292,0.003891,8.654985,0.055195,0.006917,8.847195,0.061729,0.007971,10.089081,0.037653,0.004077,12.819075,0.017958,0.001854,8.697933,0.024042,0.003679,0.8165,0,NaN,28,1.85,8,1.609000e+08,0.2,1,-20.28363,-21.12927,-21.52369,9.31240,9.27122,9.34644,9.33645,1.23468,-8.087,22.659295,22.526452,22.355068,22.042977,21.503010,21.930007,21.556834,21.532986,21.390371,21.130358,21.551460,0.132843,0.171384,0.312092,0.539966,-0.426997,0.373173,0.023848,57552,0.032255,57552,b'3962782953',150.72652,1.79573,150.72652,1.79573,1,0.82589,4,97,29,2023,1,393505,150.726503,1.795736,535,150.726526,1.795723,334248,150.726575,1.795797,495080,150.72654,1.795808,0.7747,0
1,670,149.730259,1.959835,0,1.941518,0.005379,0.003476,1.863746,0.004178,0.002408,2.359093,0.005581,0.002440,3.347864,0.005832,0.001883,4.712861,0.009339,0.002130,5.044670,0.019908,0.004038,5.936375,0.013675,0.002495,7.337306,0.017588,0.002736,9.483263,0.026426,0.003042,13.707825,0.014322,0.001382,12.806878,0.012845,0.001335,1.1792,0,NaN,30,1.02,10,2.273000e+08,0.3,0,-20.49749,-21.89549,-22.47231,9.79095,9.75692,9.83047,9.80739,1.80017,-7.953,23.179647,23.224033,22.968137,22.588081,22.216789,22.142918,21.966197,21.736158,21.457606,21.057579,21.131392,-0.044386,0.255896,0.380057,0.371292,0.073870,0.176721,0.230038,54582,0.166286,54582,b'3962783555',149.73022,1.95986,149.73022,1.95986,1,1.16549,4,97,29,2023,1,562463,149.730243,1.959851,670,149.730259,1.959835,437916,149.730268,1.959866,816008,149.73022,1.959921,1.1465,0
2,692,149.642181,2.379960,0,1.315313,0.006756,0.006467,4.484449,0.005892,0.001415,13.213070,0.007819,0.000612,24.048380,0.008534,0.000384,35.020987,0.013526,0.000416,40.134805,0.026583,0.000678,66.579523,0.061359,0.000999,98.679499,0.078428,0.000908,133.665136,0.058089,0.000475,81.719072,0.032358,0.000524,83.615927,0.028347,0.000451,0.3269,0,NaN,29,0.29,9,5.000000e+09,0.4,1,-16.72223,-20.57175,-21.94677,10.35539,10.31772,10.39305,10.38281,0.19029,-10.190,23.602427,22.270727,21.097491,20.447285,20.039179,19.891197,19.341648,18.914433,18.584955,19.119191,19.094277,1.331700,1.173237,0.650205,0.408106,0.147982,0.549549,0.427216,134345,0.069986,134345,b'174323',149.64220,2.37996,149.64217,2.37995,1,0.30572,4,97,105,2018,1,1020227,149.642181,2.379958,692,149.642181,2.379960,715344,149.642185,2.379964,1306281,149.64216,2.379982,0.3098,0
3,1143,150.589033,1.889879,0,0.063290,0.003653,0.072636,0.254108,0.004009,0.016990,1.155916,0.005216,0.004662,4.069362,0.005549,0.001476,6.650530,0.009029,0.001461,8.661979,0.019250,0.002276,14.262715,0.022103,0.001680,23.028622,0.020338,0.001009,32.456397,0.027529,0.000926,NaN,NaN,Na

In [4]:
data

,ID,RA,DEC,FLAG_COMBINED,CFHT_u_FLUX,CFHT_u_FLUXERR,CFHT_u_MAGERR,HSC_g_FLUX,HSC_g_FLUXERR,HSC_g_MAGERR,HSC_r_FLUX,HSC_r_FLUXERR,HSC_r_MAGERR,HSC_i_FLUX,HSC_i_FLUXERR,HSC_i_MAGERR,HSC_z_FLUX,HSC_z_FLUXERR,HSC_z_MAGERR,HSC_y_FLUX,HSC_y_FLUXERR,HSC_y_MAGERR,UVISTA_J_FLUX,UVISTA_J_FLUXERR,UVISTA_J_MAGERR,UVISTA_H_FLUX,UVISTA_H_FLUXERR,UVISTA_H_MAGERR,UVISTA_Ks_FLUX,UVISTA_Ks_FLUXERR,UVISTA_Ks_MAGERR,IRAC_CH1_FLUX,IRAC_CH1_FLUXERR,IRAC_CH1_MAGERR,IRAC_CH2_FLUX,IRAC_CH2_FLUXERR,IRAC_CH2_MAGERR,lp_photoz,lp_type,lp_zp_2,lp_NbFilt,lp_zq,lp_model,lp_age,lp_dust,lp_Attenuation,lp_MNUV,lp_MR,lp_MJ,lp_mass_med,lp_mass_med_min68,lp_mass_med_max68,lp_mass_best,lp_SFR_med,lp_sSFR_best,CFHT_u_MAG,HSC_g_MAG,HSC_r_MAG,HSC_i_MAG,HSC_z_MAG,HSC_y_MAG,UVISTA_J_MAG,UVISTA_H_MAG,UVISTA_Ks_MAG,IRAC_CH1_MAG,IRAC_CH2_MAG,u-g,g-r,r-i,i-z,z-y,y-J,J-H
7,12,150.476809,2.331511,0,0.432424,0.006043,0.017385,0.367684,0.004799,0.013929,0.539085,0.006673,0.012705,0.913139,0.006940,0.008187,0.995689,0.010382,0.011179,1.010389,0.021430,0.021652,1.176385,0.018240,0.016771,1.238664,0.023098,0.021265,1.273614,0.031158,0.026693,0.793401,0.012663,0.021113,0.623383,0.011264,0.024039,0.7070,0,NaN,30,0.00,5,9.048000e+08,0.1,1,-17.59872,-18.85507,-18.93651,8.69380,8.65194,8.74706,8.64422,-0.18369,-8.815,24.810225,24.986313,24.570856,23.998658,23.904690,23.888779,23.723626,23.667616,23.637405,24.151268,24.413112,-0.176089,0.415457,0.572198,0.093968,0.015911,0.165153,0.056011
39,73,149.468025,1.625609,0,1.057806,0.006321,0.007477,1.301316,0.007003,0.005769,2.870638,0.011023,0.003954,3.669017,0.012785,0.003763,4.321371,0.018494,0.004597,5.147967,0.045866,0.009110,5.366495,0.022263,0.004492,5.903986,0.031370,0.006063,6.666023,0.046399,0.007598,4.142945,0.036884,0.011779,4.426406,0.054762,0.016462,0.4290,0,NaN,30,0.34,9,1.609000e+09,0.2,1,-17.40718,-19.16920,-19.50800,9.06086,9.01925,9.10800,9.01447,0.08064,-8.977,23.838985,23.614043,22.755054,22.488626,22.310946,22.120911,22.075773,21.972137,21.840333,22.356727,22.284872,0.224942,0.858989,0.266428,0.177680,0.190035,0.045137,0.103637
40,74,150.488035,1.875422,0,0.941051,0.003713,0.004937,1.032136,0.004159,0.004320,1.066959,0.005443,0.005254,1.123350,0.005748,0.005525,1.208387,0.008979,0.007981,1.734654,0.019518,0.011504,2.413042,0.014805,0.006643,2.547439,0.017875,0.008007,2.257407,0.024673,0.011931,2.951406,0.013003,0.005829,3.054308,0.014661,0.006387,1.5087,0,NaN,29,2.23,10,9.048000e+08,0.1,1,-20.37010,-21.24365,-21.42674,9.49598,9.44923,9.54398,9.50023,0.99993,-8.536,23.965967,23.865658,23.829631,23.773713,23.694485,23.301968,22.943588,22.884741,23.015975,22.724928,22.687718,0.100310,0.036027,0.055918,0.079228,0.392517,0.358380,0.058847
46,85,150.476436,2.334178,0,0.685237,0.005227,0.009492,0.669388,0.004202,0.006701,0.744438,0.005884,0.008114,1.119968,0.005803,0.005582,1.484551,0.008772,0.006335,1.619744,0.018884,0.011903,1.782537,0.015855,0.009621,1.867551,0.020190,0.012329,2.274360,0.027033,0.012969,2.573668,0.012908,0.006634,1.992754,0.011691,0.007805,1.0800,0,NaN,29,2.19,5,2.273000e+08,0.2,0,-19.16843,-20.47925,-20.51206,9.14079,9.10236,9.18581,9.12799,0.90000,-8.234,24.310399,24.335805,24.220428,23.776986,23.471012,23.376384,23.272404,23.221819,23.007852,22.873619,23.151366,-0.025406,0.115376,0.443442,0.305974,0.094628,0.103980,0.050585
57,104,149.475722,1.626313,0,0.389645,0.004690,0.015066,0.377765,0.005186,0.014719,0.468260,0.008271,0.018192,0.582758,0.009130,0.016919,1.032440,0.013830,0.014389,1.231677,0.034203,0.028396,1.327044,0.017144,0.013989,1.671119,0.023782,0.016241,1.823997,0.035115,0.021015,2.302721,0.033234,0.019095,2.590386,0.044562,0.022890,1.2817,0,NaN,29,1.01,7,5.709000e+08,0.2,0,-18.94804,-20.47051,-20.73870,9.42859,9.36886,9.49659,9.40345,0.76785,-8.597,24.923327,24.956946,24.723783,24.486279,23.865338,23.673758,23.592787,23.342482,23.247440,22.994396,22.866589,-0.033620,0.233163,0.237504,0.620941,0.191580,0.080971,0.250305
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...

In [ ]:
#data = data[~data['ID'].isin(spec95[spec95['specz']<0.01]['ID'])]
#spec95 = spec95[spec95['specz']>0.01]
#spec95

,ID,RA,DEC,FLAG_COMBINED,CFHT_u_FLUX,CFHT_u_FLUXERR,CFHT_u_MAGERR,HSC_g_FLUX,HSC_g_FLUXERR,HSC_g_MAGERR,HSC_r_FLUX,HSC_r_FLUXERR,HSC_r_MAGERR,HSC_i_FLUX,HSC_i_FLUXERR,HSC_i_MAGERR,HSC_z_FLUX,HSC_z_FLUXERR,HSC_z_MAGERR,HSC_y_FLUX,HSC_y_FLUXERR,HSC_y_MAGERR,UVISTA_J_FLUX,UVISTA_J_FLUXERR,UVISTA_J_MAGERR,UVISTA_H_FLUX,UVISTA_H_FLUXERR,UVISTA_H_MAGERR,UVISTA_Ks_FLUX,UVISTA_Ks_FLUXERR,UVISTA_Ks_MAGERR,IRAC_CH1_FLUX,IRAC_CH1_FLUXERR,IRAC_CH1_MAGERR,IRAC_CH2_FLUX,IRAC_CH2_FLUXERR,IRAC_CH2_MAGERR,lp_photoz,lp_type,lp_zp_2,lp_NbFilt,lp_zq,lp_model,lp_age,lp_dust,lp_Attenuation,lp_MNUV,lp_MR,lp_MJ,lp_mass_med,lp_mass_med_min68,lp_mass_med_max68,lp_mass_best,lp_SFR_med,lp_sSFR_best,CFHT_u_MAG,HSC_g_MAG,HSC_r_MAG,HSC_i_MAG,HSC_z_MAG,HSC_y_MAG,UVISTA_J_MAG,UVISTA_H_MAG,UVISTA_Ks_MAG,IRAC_CH1_MAG,IRAC_CH2_MAG,u-g,g-r,r-i,i-z,z-y,y-J,J-H,nearest_cat1_ID,nearest_cat1_sep,Id_specz,Id_original,ra_original,dec_original,ra_corrected,dec_corrected,Priority,specz,flag,Confidence_level,survey,compilation_year,public_or_private,Id_COS20_Classic,ra_COS20_Classic,dec_COS20_Classic,Id_COS20_Farmer,ra_COS20_Farmer,dec_COS20_Farmer,Id_COSMOS15,ra_COSMOS15,dec_COSMOS15,Id_COSMOS09,ra_COSMOS09,dec_COSMOS09,photoz,photoz_type
0,535,150.726526,1.795723,0,3.135322,0.004911,0.001979,3.543392,0.005600,0.001707,4.149280,0.006839,0.001707,5.531073,0.006686,0.001311,9.094858,0.011055,0.001310,6.137582,0.023292,0.003891,8.654985,0.055195,0.006917,8.847195,0.061729,0.007971,10.089081,0.037653,0.004077,12.819075,0.017958,0.001854,8.697933,0.024042,0.003679,0.8165,0,NaN,28,1.85,8,1.609000e+08,0.2,1,-20.28363,-21.12927,-21.52369,9.31240,9.27122,9.34644,9.33645,1.23468,-8.087,22.659295,22.526452,22.355068,22.042977,21.503010,21.930007,21.556834,21.532986,21.390371,21.130358,21.551460,0.132843,0.171384,0.312092,0.539966,-0.426997,0.373173,0.023848,57552,0.032255,57552,b'3962782953',150.72652,1.79573,150.72652,1.79573,1,0.82589,4,97,29,2023,1,393505,150.726503,1.795736,535,150.726526,1.795723,334248,150.726575,1.795797,495080,150.72654,1.795808,0.7747,0
1,670,149.730259,1.959835,0,1.941518,0.005379,0.003476,1.863746,0.004178,0.002408,2.359093,0.005581,0.002440,3.347864,0.005832,0.001883,4.712861,0.009339,0.002130,5.044670,0.019908,0.004038,5.936375,0.013675,0.002495,7.337306,0.017588,0.002736,9.483263,0.026426,0.003042,13.707825,0.014322,0.001382,12.806878,0.012845,0.001335,1.1792,0,NaN,30,1.02,10,2.273000e+08,0.3,0,-20.49749,-21.89549,-22.47231,9.79095,9.75692,9.83047,9.80739,1.80017,-7.953,23.179647,23.224033,22.968137,22.588081,22.216789,22.142918,21.966197,21.736158,21.457606,21.057579,21.131392,-0.044386,0.255896,0.380057,0.371292,0.073870,0.176721,0.230038,54582,0.166286,54582,b'3962783555',149.73022,1.95986,149.73022,1.95986,1,1.16549,4,97,29,2023,1,562463,149.730243,1.959851,670,149.730259,1.959835,437916,149.730268,1.959866,816008,149.73022,1.959921,1.1465,0
2,692,149.642181,2.379960,0,1.315313,0.006756,0.006467,4.484449,0.005892,0.001415,13.213070,0.007819,0.000612,24.048380,0.008534,0.000384,35.020987,0.013526,0.000416,40.134805,0.026583,0.000678,66.579523,0.061359,0.000999,98.679499,0.078428,0.000908,133.665136,0.058089,0.000475,81.719072,0.032358,0.000524,83.615927,0.028347,0.000451,0.3269,0,NaN,29,0.29,9,5.000000e+09,0.4,1,-16.72223,-20.57175,-21.94677,10.35539,10.31772,10.39305,10.38281,0.19029,-10.190,23.602427,22.270727,21.097491,20.447285,20.039179,19.891197,19.341648,18.914433,18.584955,19.119191,19.094277,1.331700,1.173237,0.650205,0.408106,0.147982,0.549549,0.427216,134345,0.069986,134345,b'174323',149.64220,2.37996,149.64217,2.37995,1,0.30572,4,97,105,2018,1,1020227,149.642181,2.379958,692,149.642181,2.379960,715344,149.642185,2.379964,1306281,149.64216,2.379982,0.3098,0
3,1143,150.589033,1.889879,0,0.063290,0.003653,0.072636,0.254108,0.004009,0.016990,1.155916,0.005216,0.004662,4.069362,0.005549,0.001476,6.650530,0.009029,0.001461,8.661979,0.019250,0.002276,14.262715,0.022103,0.001680,23.028622,0.020338,0.001009,32.456397,0.027529,0.000926,NaN,NaN,Na

In [7]:
# path for saving SOM files
path = '/Users/finianashmead/Desktop/NEWMANGROUP/'
directory = 'path_pck_manhattan'
path_pck = Path(path+directory)

In [7]:
reload(umapz2)

<module 'umapz2' from '/Users/finianashmead/Desktop/NEWMANGROUP/umapz2.py'>

In [10]:
path_pck

PosixPath('/Users/finianashmead/Desktop/NEWMANGROUP/path_pck_manhattan')

In [7]:
n_embeddings = 25
for i in range(n_embeddings):
    print(i)


0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24


In [ ]:
size=(75,150)
sigma=1
learning_rate=0.8

n_embeddings = 25
for i in range(n_embeddings+1):
    print(i)
    data_i = umapz2.train_som(data, size, sigma, learning_rate, n_iter=2_000_000, rs=i, path_pck=path_pck, metric='manhattan')
    data['SOMrs'+str(i)+'-1'] = data_i['SOM_'+str(size[0])+'_'+str(size[1])+'_s'+str(sigma)+'_lr'+str(learning_rate)+'-1']
    data['SOMrs'+str(i)+'-2'] = data_i['SOM_'+str(size[0])+'_'+str(size[1])+'_s'+str(sigma)+'_lr'+str(learning_rate)+'-2']

In [14]:
#data.to_parquet('som_rs25_maps_manhattan_260509.parquet')

hi

In [18]:
data

,ID,RA,DEC,FLAG_COMBINED,CFHT_u_FLUX,CFHT_u_FLUXERR,CFHT_u_MAGERR,HSC_g_FLUX,HSC_g_FLUXERR,HSC_g_MAGERR,HSC_r_FLUX,HSC_r_FLUXERR,HSC_r_MAGERR,HSC_i_FLUX,HSC_i_FLUXERR,HSC_i_MAGERR,HSC_z_FLUX,HSC_z_FLUXERR,HSC_z_MAGERR,HSC_y_FLUX,HSC_y_FLUXERR,HSC_y_MAGERR,UVISTA_J_FLUX,UVISTA_J_FLUXERR,UVISTA_J_MAGERR,UVISTA_H_FLUX,UVISTA_H_FLUXERR,UVISTA_H_MAGERR,UVISTA_Ks_FLUX,UVISTA_Ks_FLUXERR,UVISTA_Ks_MAGERR,IRAC_CH1_FLUX,IRAC_CH1_FLUXERR,IRAC_CH1_MAGERR,IRAC_CH2_FLUX,IRAC_CH2_FLUXERR,IRAC_CH2_MAGERR,lp_photoz,lp_type,lp_zp_2,lp_NbFilt,lp_zq,lp_model,lp_age,lp_dust,lp_Attenuation,lp_MNUV,lp_MR,lp_MJ,lp_mass_med,lp_mass_med_min68,lp_mass_med_max68,lp_mass_best,lp_SFR_med,lp_sSFR_best,CFHT_u_MAG,HSC_g_MAG,HSC_r_MAG,HSC_i_MAG,HSC_z_MAG,HSC_y_MAG,UVISTA_J_MAG,UVISTA_H_MAG,UVISTA_Ks_MAG,IRAC_CH1_MAG,IRAC_CH2_MAG,u-g,g-r,r-i,i-z,z-y,y-J,J-H,SOMrs0-1,SOMrs0-2,SOMrs1-1,SOMrs1-2,SOMrs2-1,SOMrs2-2,SOMrs3-1,SOMrs3-2,SOMrs4-1,SOMrs4-2,SOMrs5-1,SOMrs5-2,SOMrs6-1,SOMrs6-2,SOMrs7-1,SOMrs7-2,SOMrs8-1,SOMrs8-2,SOMrs9-1,SOMrs9-2,SOMrs10-1,SOMrs10-2,SOMrs11-1,SOMrs11-2,SOMrs12-1,SOMrs12-2,SOMrs13-1,SOMrs13-2,SOMrs14-1,SOMrs14-2,SOMrs15-1,SOMrs15-2,SOMrs16-1,SOMrs16-2,SOMrs17-1,SOMrs17-2,SOMrs18-1,SOMrs18-2,SOMrs19-1,SOMrs19-2,SOMrs20-1,SOMrs20-2,SOMrs21-1,SOMrs21-2,SOMrs22-1,SOMrs22-2,SOMrs23-1,SOMrs23-2,SOMrs24-1,SOMrs24-2,SOMrs25-1,SOMrs25-2
7,12,150.476809,2.331511,0,0.432424,0.006043,0.017385,0.367684,0.004799,0.013929,0.539085,0.006673,0.012705,0.913139,0.006940,0.008187,0.995689,0.010382,0.011179,1.010389,0.021430,0.021652,1.176385,0.018240,0.016771,1.238664,0.023098,0.021265,1.273614,0.031158,0.026693,0.793401,0.012663,0.021113,0.623383,0.011264,0.024039,0.7070,0,NaN,30,0.00,5,9.048000e+08,0.1,1,-17.59872,-18.85507,-18.93651,8.69380,8.65194,8.74706,8.64422,-0.18369,-8.815,24.810225,24.986313,24.570856,23.998658,23.904690,23.888779,23.723626,23.667616,23.637405,24.151268,24.413112,-0.176089,0.415457,0.572198,0.093968,0.015911,0.165153,0.056011,13,62,9,81,24,83,28,96,47,93,46,81,40,87,12,22,39,83,14,76,20,93,47,50,45,90,35,17,2,46,41,83,3,36,19,60,12,69,4,41,39,34,10,15,37,84,44,18,12,13,47,31
39,73,149.468025,1.625609,0,1.057806,0.006321,0.007477,1.301316,0.007003,0.005769,2.870638,0.011023,0.003954,3.669017,0.012785,0.003763,4.321371,0.018494,0.004597,5.147967,0.045866,0.009110,5.366495,0.022263,0.004492,5.903986,0.031370,0.006063,6.666023,0.046399,0.007598,4.142945,0.036884,0.011779,4.426406,0.054762,0.016462,0.4290,0,NaN,30,0.34,9,1.609000e+09,0.2,1,-17.40718,-19.16920,-19.50800,9.06086,9.01925,9.10800,9.01447,0.08064,-8.977,23.838985,23.614043,22.755054,22.488626,22.310946,22.120911,22.075773,21.972137,21.840333,22.356727,22.284872,0.224942,0.858989,0.266428,0.177680,0.190035,0.045137,0.103637,11,21,25,31,6,55,32,46,42,37,21,35,31,49,42,53,11,37,7,67,11,91,5,66,47,57,45,5,8,79,47,97,16,42,24,3,18,58,7,79,9,49,5,1,25,77,38,38,43,21,45,10
40,74,150.488035,1.875422,0,0.941051,0.003713,0.004937,1.032136,0.004159,0.004320,1.066959,0.005443,0.005254,1.123350,0.005748,0.005525,1.208387,0.008979,0.007981,1.734654,0.019518,0.011504,2.413042,0.014805,0.006643,2.547439,0.017875,0.008007,2.257407,0.024673,0.011931,2.951406,0.013003,0.005829,3.054308,0.014661,0.006387,1.5087,0,NaN,29,2.23,10,9.048000e+08,0.1,1,-20.37010,-21.24365,-21.42674,9.49598,9.44923,9.54398,9.50023,0.99993,-8.536,23.965967,23.865658,23.829631,23.773713,23.694485,23.301968,22.943588,22.884741,23.015975,22.724928,22.687718,0.100310,0.036027,0.055918,0.079228,0.392517,0.358380,0.058847,43,95,46,91,25,95,17,70,5,93,6,96,0,95,36,31,49,24,29,84,46,80,16,51,22,96,4,92,14,1,14,47,12,90,8,94,43,55,29,37,34,7,25,19,3,35,36,72,2,47,28,20
46,85,150.476436,2.334178,0,0.685237,0.005227,0.009492,0.669388,0.004202,0.006701,0.744438,0.005884,0.008114,1.119968,0.005803,0.005582,1.484551,0.008772,0.006335,1.619744,0.018884,0.011903,1.782537,0.015855,0.009621,1.867551,0.020190,0.012329,2.274360,0.027033,0.012969,2.573668,0.012908,0.006634,1.992754,0.011691,0.007805,1.0800,0,NaN,29,2.19,5,2.273000e+08,0.2,0,

In [8]:
path_pck

PosixPath('/Users/finianashmead/Desktop/NEWMANGROUP/path_pck_manhattan')

In [ ]:
#sizes=[(8,16), (12,24), (18,36), (25,50), (35,70), (50,100), (75,150)]
sizes=[(69,138),(75,150),(80,160)]

sigmas = [0.85, 1, 1.15]
learning_rates = [0.68, 0.8, 0.92]

data_soms = umapz2.train_soms(data, sizes, sigmas, learning_rates, n_iter=2_000_000, rs=42, path_pck=path_pck, metric='manhattan')

In [10]:
data_soms

,ID,RA,DEC,FLAG_COMBINED,CFHT_u_FLUX,CFHT_u_FLUXERR,CFHT_u_MAGERR,HSC_g_FLUX,HSC_g_FLUXERR,HSC_g_MAGERR,HSC_r_FLUX,HSC_r_FLUXERR,HSC_r_MAGERR,HSC_i_FLUX,HSC_i_FLUXERR,HSC_i_MAGERR,HSC_z_FLUX,HSC_z_FLUXERR,HSC_z_MAGERR,HSC_y_FLUX,HSC_y_FLUXERR,HSC_y_MAGERR,UVISTA_J_FLUX,UVISTA_J_FLUXERR,UVISTA_J_MAGERR,UVISTA_H_FLUX,UVISTA_H_FLUXERR,UVISTA_H_MAGERR,UVISTA_Ks_FLUX,UVISTA_Ks_FLUXERR,UVISTA_Ks_MAGERR,IRAC_CH1_FLUX,IRAC_CH1_FLUXERR,IRAC_CH1_MAGERR,IRAC_CH2_FLUX,IRAC_CH2_FLUXERR,IRAC_CH2_MAGERR,lp_photoz,lp_type,lp_zp_2,lp_NbFilt,lp_zq,lp_model,lp_age,lp_dust,lp_Attenuation,lp_MNUV,lp_MR,lp_MJ,lp_mass_med,lp_mass_med_min68,lp_mass_med_max68,lp_mass_best,lp_SFR_med,lp_sSFR_best,CFHT_u_MAG,HSC_g_MAG,HSC_r_MAG,HSC_i_MAG,HSC_z_MAG,HSC_y_MAG,UVISTA_J_MAG,UVISTA_H_MAG,UVISTA_Ks_MAG,IRAC_CH1_MAG,IRAC_CH2_MAG,u-g,g-r,r-i,i-z,z-y,y-J,J-H,SOM_69_138_s0.85_lr0.68-1,SOM_69_138_s0.85_lr0.68-2,SOM_69_138_s0.85_lr0.8-1,SOM_69_138_s0.85_lr0.8-2,SOM_69_138_s0.85_lr0.92-1,SOM_69_138_s0.85_lr0.92-2,SOM_69_138_s1_lr0.68-1,SOM_69_138_s1_lr0.68-2,SOM_69_138_s1_lr0.8-1,SOM_69_138_s1_lr0.8-2,SOM_69_138_s1_lr0.92-1,SOM_69_138_s1_lr0.92-2,SOM_69_138_s1.15_lr0.68-1,SOM_69_138_s1.15_lr0.68-2,SOM_69_138_s1.15_lr0.8-1,SOM_69_138_s1.15_lr0.8-2,SOM_69_138_s1.15_lr0.92-1,SOM_69_138_s1.15_lr0.92-2,SOM_75_150_s0.85_lr0.68-1,SOM_75_150_s0.85_lr0.68-2,SOM_75_150_s0.85_lr0.8-1,SOM_75_150_s0.85_lr0.8-2,SOM_75_150_s0.85_lr0.92-1,SOM_75_150_s0.85_lr0.92-2,SOM_75_150_s1_lr0.68-1,SOM_75_150_s1_lr0.68-2,SOM_75_150_s1_lr0.8-1,SOM_75_150_s1_lr0.8-2,SOM_75_150_s1_lr0.92-1,SOM_75_150_s1_lr0.92-2,SOM_75_150_s1.15_lr0.68-1,SOM_75_150_s1.15_lr0.68-2,SOM_75_150_s1.15_lr0.8-1,SOM_75_150_s1.15_lr0.8-2,SOM_75_150_s1.15_lr0.92-1,SOM_75_150_s1.15_lr0.92-2,SOM_80_160_s0.85_lr0.68-1,SOM_80_160_s0.85_lr0.68-2,SOM_80_160_s0.85_lr0.8-1,SOM_80_160_s0.85_lr0.8-2,SOM_80_160_s0.85_lr0.92-1,SOM_80_160_s0.85_lr0.92-2,SOM_80_160_s1_lr0.68-1,SOM_80_160_s1_lr0.68-2,SOM_80_160_s1_lr0.8-1,SOM_80_160_s1_lr0.8-2,SOM_80_160_s1_lr0.92-1,SOM_80_160_s1_lr0.92-2,SOM_80_160_s1.15_lr0.68-1,SOM_80_160_s1.15_lr0.68-2,SOM_80_160_s1.15_lr0.8-1,SOM_80_160_s1.15_lr0.8-2,SOM_80_160_s1.15_lr0.92-1,SOM_80_160_s1.15_lr0.92-2
7,12,150.476809,2.331511,0,0.432424,0.006043,0.017385,0.367684,0.004799,0.013929,0.539085,0.006673,0.012705,0.913139,0.006940,0.008187,0.995689,0.010382,0.011179,1.010389,0.021430,0.021652,1.176385,0.018240,0.016771,1.238664,0.023098,0.021265,1.273614,0.031158,0.026693,0.793401,0.012663,0.021113,0.623383,0.011264,0.024039,0.7070,0,NaN,30,0.00,5,9.048000e+08,0.1,1,-17.59872,-18.85507,-18.93651,8.69380,8.65194,8.74706,8.64422,-0.18369,-8.815,24.810225,24.986313,24.570856,23.998658,23.904690,23.888779,23.723626,23.667616,23.637405,24.151268,24.413112,-0.176089,0.415457,0.572198,0.093968,0.015911,0.165153,0.056011,0,129,0,12,13,80,43,11,28,10,43,10,47,116,56,53,34,124,62,69,74,29,38,8,56,125,53,128,64,68,53,110,11,147,13,126,43,80,12,108,4,95,43,89,69,27,3,100,48,89,27,156,67,99
39,73,149.468025,1.625609,0,1.057806,0.006321,0.007477,1.301316,0.007003,0.005769,2.870638,0.011023,0.003954,3.669017,0.012785,0.003763,4.321371,0.018494,0.004597,5.147967,0.045866,0.009110,5.366495,0.022263,0.004492,5.903986,0.031370,0.006063,6.666023,0.046399,0.007598,4.142945,0.036884,0.011779,4.426406,0.054762,0.016462,0.4290,0,NaN,30,0.34,9,1.609000e+09,0.2,1,-17.40718,-19.16920,-19.50800,9.06086,9.01925,9.10800,9.01447,0.08064,-8.977,23.838985,23.614043,22.755054,22.488626,22.310946,22.120911,22.075773,21.972137,21.840333,22.356727,22.284872,0.224942,0.858989,0.266428,0.177680,0.190035,0.045137,0.103637,35,25,26,6,59,49,30,2,60,52,46,99,21,4,38,7,31,16,29,111,70,131,24,26,26,18,29,63,48,62,61,127,45,65,34,56,74,19,28,84,24,96,16,88,16,91,20,83,16,84,44,105,77,17
40,74,150.488035,1.875422,0,0.941051,0.003713,0.004937,1.032136,0.004159,0.004320,1.066959,0.005443,0.005254,1.123350,0.005748,0.005525,1.208387,0.008979,0.007981,1.734654,0.019518,0.011504,2.413042,0.014805,0.006643,2.547439,0.017875,0.008007,2.257407,0.024673,0.0119

In [11]:
#data_soms.to_parquet('som_embeddings_manhattan_260507.parquet')

In [18]:
splits_soms = umapz2.data_split(photcat=data_soms, speccat=spec95, valid_size=len(spec95), rs=41)
splits_soms.keys()

dict_keys(['valid', 'train_lp', 'test_lp', 'train_spec', 'test_spec'])

In [ ]:
sizes=[(8,16), (12,24), (18,36), (25,50), (35,70), (50,100), (75,150)]
#sizes=[(75,)]

sigmas = [0.5, 1, 2, 4]
learning_rates = [0.8]

for size in sizes:
        n_som_xy = size
        for sigma in sigmas:
            for lr in learning_rates:
                map = 'SOM_'+str(n_som_xy[0])+'_'+str(n_som_xy[1])+'_s'+str(sigma)+'_lr'+str(lr)
                umapz2.plot_som_valid(full_df=data_soms, valid_df=splits_soms['valid'], 
                                      map=map, size=size, 
                                      filename=map+'_n2M_manhattan_260507.png');

In [9]:
#data_soms.to_parquet('som_embeddings_manhattan_260507.parquet')

data_soms

In [10]:
#data_soms.to_parquet('som_embeddings2M_260409.parquet')

In [7]:
data

,ID,RA,DEC,FLAG_COMBINED,CFHT_u_FLUX,CFHT_u_FLUXERR,CFHT_u_MAGERR,HSC_g_FLUX,HSC_g_FLUXERR,HSC_g_MAGERR,HSC_r_FLUX,HSC_r_FLUXERR,HSC_r_MAGERR,HSC_i_FLUX,HSC_i_FLUXERR,HSC_i_MAGERR,HSC_z_FLUX,HSC_z_FLUXERR,HSC_z_MAGERR,HSC_y_FLUX,HSC_y_FLUXERR,HSC_y_MAGERR,UVISTA_J_FLUX,UVISTA_J_FLUXERR,UVISTA_J_MAGERR,UVISTA_H_FLUX,UVISTA_H_FLUXERR,UVISTA_H_MAGERR,UVISTA_Ks_FLUX,UVISTA_Ks_FLUXERR,UVISTA_Ks_MAGERR,IRAC_CH1_FLUX,IRAC_CH1_FLUXERR,IRAC_CH1_MAGERR,IRAC_CH2_FLUX,IRAC_CH2_FLUXERR,IRAC_CH2_MAGERR,lp_photoz,lp_type,lp_zp_2,lp_NbFilt,lp_zq,lp_model,lp_age,lp_dust,lp_Attenuation,lp_MNUV,lp_MR,lp_MJ,lp_mass_med,lp_mass_med_min68,lp_mass_med_max68,lp_mass_best,lp_SFR_med,lp_sSFR_best,CFHT_u_MAG,HSC_g_MAG,HSC_r_MAG,HSC_i_MAG,HSC_z_MAG,HSC_y_MAG,UVISTA_J_MAG,UVISTA_H_MAG,UVISTA_Ks_MAG,IRAC_CH1_MAG,IRAC_CH2_MAG,u-g,g-r,r-i,i-z,z-y,y-J,J-H
7,12,150.476809,2.331511,0,0.432424,0.006043,0.017385,0.367684,0.004799,0.013929,0.539085,0.006673,0.012705,0.913139,0.006940,0.008187,0.995689,0.010382,0.011179,1.010389,0.021430,0.021652,1.176385,0.018240,0.016771,1.238664,0.023098,0.021265,1.273614,0.031158,0.026693,0.793401,0.012663,0.021113,0.623383,0.011264,0.024039,0.7070,0,NaN,30,0.00,5,9.048000e+08,0.1,1,-17.59872,-18.85507,-18.93651,8.69380,8.65194,8.74706,8.64422,-0.18369,-8.815,24.810225,24.986313,24.570856,23.998658,23.904690,23.888779,23.723626,23.667616,23.637405,24.151268,24.413112,-0.176089,0.415457,0.572198,0.093968,0.015911,0.165153,0.056011
39,73,149.468025,1.625609,0,1.057806,0.006321,0.007477,1.301316,0.007003,0.005769,2.870638,0.011023,0.003954,3.669017,0.012785,0.003763,4.321371,0.018494,0.004597,5.147967,0.045866,0.009110,5.366495,0.022263,0.004492,5.903986,0.031370,0.006063,6.666023,0.046399,0.007598,4.142945,0.036884,0.011779,4.426406,0.054762,0.016462,0.4290,0,NaN,30,0.34,9,1.609000e+09,0.2,1,-17.40718,-19.16920,-19.50800,9.06086,9.01925,9.10800,9.01447,0.08064,-8.977,23.838985,23.614043,22.755054,22.488626,22.310946,22.120911,22.075773,21.972137,21.840333,22.356727,22.284872,0.224942,0.858989,0.266428,0.177680,0.190035,0.045137,0.103637
40,74,150.488035,1.875422,0,0.941051,0.003713,0.004937,1.032136,0.004159,0.004320,1.066959,0.005443,0.005254,1.123350,0.005748,0.005525,1.208387,0.008979,0.007981,1.734654,0.019518,0.011504,2.413042,0.014805,0.006643,2.547439,0.017875,0.008007,2.257407,0.024673,0.011931,2.951406,0.013003,0.005829,3.054308,0.014661,0.006387,1.5087,0,NaN,29,2.23,10,9.048000e+08,0.1,1,-20.37010,-21.24365,-21.42674,9.49598,9.44923,9.54398,9.50023,0.99993,-8.536,23.965967,23.865658,23.829631,23.773713,23.694485,23.301968,22.943588,22.884741,23.015975,22.724928,22.687718,0.100310,0.036027,0.055918,0.079228,0.392517,0.358380,0.058847
46,85,150.476436,2.334178,0,0.685237,0.005227,0.009492,0.669388,0.004202,0.006701,0.744438,0.005884,0.008114,1.119968,0.005803,0.005582,1.484551,0.008772,0.006335,1.619744,0.018884,0.011903,1.782537,0.015855,0.009621,1.867551,0.020190,0.012329,2.274360,0.027033,0.012969,2.573668,0.012908,0.006634,1.992754,0.011691,0.007805,1.0800,0,NaN,29,2.19,5,2.273000e+08,0.2,0,-19.16843,-20.47925,-20.51206,9.14079,9.10236,9.18581,9.12799,0.90000,-8.234,24.310399,24.335805,24.220428,23.776986,23.471012,23.376384,23.272404,23.221819,23.007852,22.873619,23.151366,-0.025406,0.115376,0.443442,0.305974,0.094628,0.103980,0.050585
57,104,149.475722,1.626313,0,0.389645,0.004690,0.015066,0.377765,0.005186,0.014719,0.468260,0.008271,0.018192,0.582758,0.009130,0.016919,1.032440,0.013830,0.014389,1.231677,0.034203,0.028396,1.327044,0.017144,0.013989,1.671119,0.023782,0.016241,1.823997,0.035115,0.021015,2.302721,0.033234,0.019095,2.590386,0.044562,0.022890,1.2817,0,NaN,29,1.01,7,5.709000e+08,0.2,0,-18.94804,-20.47051,-20.73870,9.42859,9.36886,9.49659,9.40345,0.76785,-8.597,24.923327,24.956946,24.723783,24.486279,23.865338,23.673758,23.592787,23.342482,23.247440,22.994396,22.866589,-0.033620,0.233163,0.237504,0.620941,0.191580,0.080971,0.250305
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...

In [8]:
# loop over n_neighbors hyperparameter arguments (nn_hps) to generate multiple embeddings

#nn_hps = [55, 60, 65, 70, 75, 80, 85]
nn_hps = [50, 55, 60, 65, 70, 75, 80, 85, 90]
#nn_hps = [68, 80, 92]

for nn_hp in nn_hps:
    emb_i = umap.UMAP(n_neighbors=nn_hp, min_dist=0.0, metric='manhattan', n_components=3, random_state=41, densmap=False).fit(data[['u-g', 'g-r', 'r-i', 'i-z', 'z-y', 'y-J', 'J-H']])
    data['UMAPnn'+str(nn_hp)+'-1'] = emb_i.embedding_[:,0]
    data['UMAPnn'+str(nn_hp)+'-2'] = emb_i.embedding_[:,1]
    data['UMAPnn'+str(nn_hp)+'-3'] = emb_i.embedding_[:,2]

/Applications/anaconda3/envs/env1/lib/python3.11/site-packages/umap/umap_.py:1943: UserWarning: n_jobs value -1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(f"n_jobs value {self.n_jobs} overridden to 1 by setting random_state. Use no seed for parallelism.")
/var/folders/zr/5945lkps2z573rnydx977g7m0000gn/T/ipykernel_4284/303788065.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['UMAPnn'+str(nn_hp)+'-1'] = emb_i.embedding_[:,0]
/var/folders/zr/5945lkps2z573rnydx977g7m0000gn/T/ipykernel_4284/303788065.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.py

In [26]:
#data.to_parquet('som_umap_uncbudget_maps_260415.parquet')

In [9]:
data

,ID,RA,DEC,FLAG_COMBINED,CFHT_u_FLUX,CFHT_u_FLUXERR,CFHT_u_MAGERR,HSC_g_FLUX,HSC_g_FLUXERR,HSC_g_MAGERR,HSC_r_FLUX,HSC_r_FLUXERR,HSC_r_MAGERR,HSC_i_FLUX,HSC_i_FLUXERR,HSC_i_MAGERR,HSC_z_FLUX,HSC_z_FLUXERR,HSC_z_MAGERR,HSC_y_FLUX,HSC_y_FLUXERR,HSC_y_MAGERR,UVISTA_J_FLUX,UVISTA_J_FLUXERR,UVISTA_J_MAGERR,UVISTA_H_FLUX,UVISTA_H_FLUXERR,UVISTA_H_MAGERR,UVISTA_Ks_FLUX,UVISTA_Ks_FLUXERR,UVISTA_Ks_MAGERR,IRAC_CH1_FLUX,IRAC_CH1_FLUXERR,IRAC_CH1_MAGERR,IRAC_CH2_FLUX,IRAC_CH2_FLUXERR,IRAC_CH2_MAGERR,lp_photoz,lp_type,lp_zp_2,lp_NbFilt,lp_zq,lp_model,lp_age,lp_dust,lp_Attenuation,lp_MNUV,lp_MR,lp_MJ,lp_mass_med,lp_mass_med_min68,lp_mass_med_max68,lp_mass_best,lp_SFR_med,lp_sSFR_best,CFHT_u_MAG,HSC_g_MAG,HSC_r_MAG,HSC_i_MAG,HSC_z_MAG,HSC_y_MAG,UVISTA_J_MAG,UVISTA_H_MAG,UVISTA_Ks_MAG,IRAC_CH1_MAG,IRAC_CH2_MAG,u-g,g-r,r-i,i-z,z-y,y-J,J-H,UMAPnn50-1,UMAPnn50-2,UMAPnn50-3,UMAPnn55-1,UMAPnn55-2,UMAPnn55-3,UMAPnn60-1,UMAPnn60-2,UMAPnn60-3,UMAPnn65-1,UMAPnn65-2,UMAPnn65-3,UMAPnn70-1,UMAPnn70-2,UMAPnn70-3,UMAPnn75-1,UMAPnn75-2,UMAPnn75-3,UMAPnn80-1,UMAPnn80-2,UMAPnn80-3,UMAPnn85-1,UMAPnn85-2,UMAPnn85-3,UMAPnn90-1,UMAPnn90-2,UMAPnn90-3
7,12,150.476809,2.331511,0,0.432424,0.006043,0.017385,0.367684,0.004799,0.013929,0.539085,0.006673,0.012705,0.913139,0.006940,0.008187,0.995689,0.010382,0.011179,1.010389,0.021430,0.021652,1.176385,0.018240,0.016771,1.238664,0.023098,0.021265,1.273614,0.031158,0.026693,0.793401,0.012663,0.021113,0.623383,0.011264,0.024039,0.7070,0,NaN,30,0.00,5,9.048000e+08,0.1,1,-17.59872,-18.85507,-18.93651,8.69380,8.65194,8.74706,8.64422,-0.18369,-8.815,24.810225,24.986313,24.570856,23.998658,23.904690,23.888779,23.723626,23.667616,23.637405,24.151268,24.413112,-0.176089,0.415457,0.572198,0.093968,0.015911,0.165153,0.056011,-1.033182,3.720988,-1.388137,-0.959336,3.825539,-1.231575,-0.881290,3.796339,-1.209693,-0.870086,3.704909,-0.993322,-0.741683,3.692303,-0.950502,-0.785397,3.743953,-0.928493,-0.760679,3.647432,-0.850287,-0.661774,3.742576,-0.734830,-0.621733,3.676099,-0.746140
39,73,149.468025,1.625609,0,1.057806,0.006321,0.007477,1.301316,0.007003,0.005769,2.870638,0.011023,0.003954,3.669017,0.012785,0.003763,4.321371,0.018494,0.004597,5.147967,0.045866,0.009110,5.366495,0.022263,0.004492,5.903986,0.031370,0.006063,6.666023,0.046399,0.007598,4.142945,0.036884,0.011779,4.426406,0.054762,0.016462,0.4290,0,NaN,30,0.34,9,1.609000e+09,0.2,1,-17.40718,-19.16920,-19.50800,9.06086,9.01925,9.10800,9.01447,0.08064,-8.977,23.838985,23.614043,22.755054,22.488626,22.310946,22.120911,22.075773,21.972137,21.840333,22.356727,22.284872,0.224942,0.858989,0.266428,0.177680,0.190035,0.045137,0.103637,0.766996,6.048308,-1.416872,0.806598,6.124864,-1.232763,0.871471,6.029686,-1.179094,0.866591,6.002512,-1.039919,0.968271,5.938120,-1.001814,0.977910,5.954481,-0.898991,0.951395,5.896547,-0.895896,1.066993,5.960493,-0.719571,1.100952,5.834060,-0.780278
40,74,150.488035,1.875422,0,0.941051,0.003713,0.004937,1.032136,0.004159,0.004320,1.066959,0.005443,0.005254,1.123350,0.005748,0.005525,1.208387,0.008979,0.007981,1.734654,0.019518,0.011504,2.413042,0.014805,0.006643,2.547439,0.017875,0.008007,2.257407,0.024673,0.011931,2.951406,0.013003,0.005829,3.054308,0.014661,0.006387,1.5087,0,NaN,29,2.23,10,9.048000e+08,0.1,1,-20.37010,-21.24365,-21.42674,9.49598,9.44923,9.54398,9.50023,0.99993,-8.536,23.965967,23.865658,23.829631,23.773713,23.694485,23.301968,22.943588,22.884741,23.015975,22.724928,22.687718,0.100310,0.036027,0.055918,0.079228,0.392517,0.358380,0.058847,0.555235,4.140800,9.900080,0.633083,4.132921,10.070456,0.533604,4.124071,10.108575,0.624591,4.132191,10.229904,0.640105,4.060228,10.181063,0.650863,4.067042,10.252739,0.659285,4.106952,10.281441,0.690439,4.038807,10.390600,0.633836,4.054846,10.377823
46,85,150.476436,2.334178,0,0.685237,0.005227,0.009492,0.669388,0.004202,0.006701,0.744438,0.005884,0.008114,1.119968,0.005803,0.005582,1.484551,0.008772,0.006335,1.619744,0.018884,0.011903,1.782537,0.015855,0.009621,1.867551,0.020190,0.012329,2.27436

In [10]:
data.to_parquet('umap_embeddings_nntest_260518.parquet')

In [9]:
umaps = pd.read_parquet('embeddings_nntest_260313.parquet')
umaps

,ID,RA,DEC,FLAG_COMBINED,CFHT_u_FLUX,CFHT_u_FLUXERR,CFHT_u_MAGERR,HSC_g_FLUX,HSC_g_FLUXERR,HSC_g_MAGERR,HSC_r_FLUX,HSC_r_FLUXERR,HSC_r_MAGERR,HSC_i_FLUX,HSC_i_FLUXERR,HSC_i_MAGERR,HSC_z_FLUX,HSC_z_FLUXERR,HSC_z_MAGERR,HSC_y_FLUX,HSC_y_FLUXERR,HSC_y_MAGERR,UVISTA_J_FLUX,UVISTA_J_FLUXERR,UVISTA_J_MAGERR,UVISTA_H_FLUX,UVISTA_H_FLUXERR,UVISTA_H_MAGERR,UVISTA_Ks_FLUX,UVISTA_Ks_FLUXERR,UVISTA_Ks_MAGERR,IRAC_CH1_FLUX,IRAC_CH1_FLUXERR,IRAC_CH1_MAGERR,IRAC_CH2_FLUX,IRAC_CH2_FLUXERR,IRAC_CH2_MAGERR,lp_photoz,lp_type,lp_zp_2,lp_NbFilt,lp_zq,lp_model,lp_age,lp_dust,lp_Attenuation,lp_MNUV,lp_MR,lp_MJ,lp_mass_med,lp_mass_med_min68,lp_mass_med_max68,lp_mass_best,lp_SFR_med,lp_sSFR_best,CFHT_u_MAG,HSC_g_MAG,HSC_r_MAG,HSC_i_MAG,HSC_z_MAG,HSC_y_MAG,UVISTA_J_MAG,UVISTA_H_MAG,UVISTA_Ks_MAG,IRAC_CH1_MAG,IRAC_CH2_MAG,u-g,g-r,r-i,i-z,z-y,y-J,J-H,UMAPnn55-1,UMAPnn55-2,UMAPnn55-3,UMAPnn60-1,UMAPnn60-2,UMAPnn60-3,UMAPnn65-1,UMAPnn65-2,UMAPnn65-3,UMAPnn70-1,UMAPnn70-2,UMAPnn70-3,UMAPnn75-1,UMAPnn75-2,UMAPnn75-3,UMAPnn80-1,UMAPnn80-2,UMAPnn80-3,UMAPnn85-1,UMAPnn85-2,UMAPnn85-3
7,12,150.476809,2.331511,0,0.432424,0.006043,0.017385,0.367684,0.004799,0.013929,0.539085,0.006673,0.012705,0.913139,0.006940,0.008187,0.995689,0.010382,0.011179,1.010389,0.021430,0.021652,1.176385,0.018240,0.016771,1.238664,0.023098,0.021265,1.273614,0.031158,0.026693,0.793401,0.012663,0.021113,0.623383,0.011264,0.024039,0.7070,0,NaN,30,0.00,5,9.048000e+08,0.1,1,-17.59872,-18.85507,-18.93651,8.69380,8.65194,8.74706,8.64422,-0.18369,-8.815,24.810225,24.986313,24.570856,23.998658,23.904690,23.888779,23.723626,23.667616,23.637405,24.151268,24.413112,-0.176089,0.415457,0.572198,0.093968,0.015911,0.165153,0.056011,-0.959336,3.825539,-1.231575,-0.881290,3.796339,-1.209693,-0.870086,3.704909,-0.993322,-0.741683,3.692303,-0.950502,-0.785397,3.743953,-0.928493,-0.760679,3.647432,-0.850287,-0.661774,3.742576,-0.734830
39,73,149.468025,1.625609,0,1.057806,0.006321,0.007477,1.301316,0.007003,0.005769,2.870638,0.011023,0.003954,3.669017,0.012785,0.003763,4.321371,0.018494,0.004597,5.147967,0.045866,0.009110,5.366495,0.022263,0.004492,5.903986,0.031370,0.006063,6.666023,0.046399,0.007598,4.142945,0.036884,0.011779,4.426406,0.054762,0.016462,0.4290,0,NaN,30,0.34,9,1.609000e+09,0.2,1,-17.40718,-19.16920,-19.50800,9.06086,9.01925,9.10800,9.01447,0.08064,-8.977,23.838985,23.614043,22.755054,22.488626,22.310946,22.120911,22.075773,21.972137,21.840333,22.356727,22.284872,0.224942,0.858989,0.266428,0.177680,0.190035,0.045137,0.103637,0.806598,6.124864,-1.232763,0.871471,6.029686,-1.179094,0.866591,6.002512,-1.039919,0.968271,5.938120,-1.001814,0.977910,5.954481,-0.898991,0.951395,5.896547,-0.895896,1.066993,5.960493,-0.719571
40,74,150.488035,1.875422,0,0.941051,0.003713,0.004937,1.032136,0.004159,0.004320,1.066959,0.005443,0.005254,1.123350,0.005748,0.005525,1.208387,0.008979,0.007981,1.734654,0.019518,0.011504,2.413042,0.014805,0.006643,2.547439,0.017875,0.008007,2.257407,0.024673,0.011931,2.951406,0.013003,0.005829,3.054308,0.014661,0.006387,1.5087,0,NaN,29,2.23,10,9.048000e+08,0.1,1,-20.37010,-21.24365,-21.42674,9.49598,9.44923,9.54398,9.50023,0.99993,-8.536,23.965967,23.865658,23.829631,23.773713,23.694485,23.301968,22.943588,22.884741,23.015975,22.724928,22.687718,0.100310,0.036027,0.055918,0.079228,0.392517,0.358380,0.058847,0.633083,4.132921,10.070456,0.533604,4.124071,10.108575,0.624591,4.132191,10.229904,0.640105,4.060228,10.181063,0.650863,4.067042,10.252739,0.659285,4.106952,10.281441,0.690439,4.038807,10.390600
46,85,150.476436,2.334178,0,0.685237,0.005227,0.009492,0.669388,0.004202,0.006701,0.744438,0.005884,0.008114,1.119968,0.005803,0.005582,1.484551,0.008772,0.006335,1.619744,0.018884,0.011903,1.782537,0.015855,0.009621,1.867551,0.020190,0.012329,2.274360,0.027033,0.012969,2.573668,0.012908,0.006634,1.992754,0.011691,0.007805,1.0800,0,NaN,29,2.19,5,2.273000e+08,0.2,0,-19.16843,-20.47925,-20.51206,9.14079,9.10236,9.18581,9.12799,0.90000,-8.234,24.310399,24.335805,24.220428,23.776986,23

In [9]:
splits_soms = umapz2.data_split(photcat=data_soms, speccat=spec95, valid_size=len(spec95), rs=41)
splits_soms.keys()

dict_keys(['valid', 'train_lp', 'test_lp', 'train_spec', 'test_spec'])

In [7]:
# PROCESSING / SPLITS
soms = pd.read_parquet('som_embeddings2M_260409.parquet')
spec_soms = pd.merge(soms.copy()[soms['ID'].isin(spec95['ID'])], spec95[['ID', 'specz']], left_on='ID', right_on='ID', how='left')
splits_soms = umapz2.data_split(photcat=soms, speccat=spec_soms, valid_size=len(spec_soms), rs=41)
valid_soms = copy.deepcopy(splits_soms['valid'])

In [19]:
# PROCESSING / SPLITS
soms = pd.read_parquet('som_rs25_maps.parquet')
spec_soms = pd.merge(soms.copy()[soms['ID'].isin(spec95['ID'])], spec95[['ID', 'specz']], left_on='ID', right_on='ID', how='left')
splits_soms = umapz2.data_split(photcat=soms, speccat=spec_soms, valid_size=len(spec_soms), rs=41)
valid_soms = copy.deepcopy(splits_soms['valid'])

In [ ]:
n_embeddings=25
for i in range(1,n_embeddings+1): 
    map = 'SOMrs'+str(i)
    umapz2.plot_som_valid(full_df=soms, valid_df=splits_soms['valid'], map=map, size=(50,100))

In [ ]:
#sizes=[(50,100)]
#sigmas = [0.5, 1, 2, 4]
#learning_rates = [0.2, 0.4, 0.8, 1.]

sizes=[(46,92), (50,100), (54,108)]
#sizes=[(75,)]

sigmas = [0.85, 1, 1.15]
learning_rates = [0.68, 0.8, 0.92]

for size in sizes:
        n_som_xy = size
        for sigma in sigmas:
            for lr in learning_rates:
                map = 'SOM_'+str(n_som_xy[0])+'_'+str(n_som_xy[1])+'_s'+str(sigma)+'_lr'+str(lr)
                umapz2.plot_som_valid(full_df=soms, valid_df=splits_soms['valid'], 
                                      map=map, size=(8,16))#, 
                                      #filename=map+'_n2M_260407.png');

In [5]:
data

,ID,RA,DEC,FLAG_COMBINED,CFHT_u_FLUX,CFHT_u_FLUXERR,CFHT_u_MAGERR,HSC_g_FLUX,HSC_g_FLUXERR,HSC_g_MAGERR,HSC_r_FLUX,HSC_r_FLUXERR,HSC_r_MAGERR,HSC_i_FLUX,HSC_i_FLUXERR,HSC_i_MAGERR,HSC_z_FLUX,HSC_z_FLUXERR,HSC_z_MAGERR,HSC_y_FLUX,HSC_y_FLUXERR,HSC_y_MAGERR,UVISTA_J_FLUX,UVISTA_J_FLUXERR,UVISTA_J_MAGERR,UVISTA_H_FLUX,UVISTA_H_FLUXERR,UVISTA_H_MAGERR,UVISTA_Ks_FLUX,UVISTA_Ks_FLUXERR,UVISTA_Ks_MAGERR,IRAC_CH1_FLUX,IRAC_CH1_FLUXERR,IRAC_CH1_MAGERR,IRAC_CH2_FLUX,IRAC_CH2_FLUXERR,IRAC_CH2_MAGERR,lp_photoz,lp_type,lp_zp_2,lp_NbFilt,lp_zq,lp_model,lp_age,lp_dust,lp_Attenuation,lp_MNUV,lp_MR,lp_MJ,lp_mass_med,lp_mass_med_min68,lp_mass_med_max68,lp_mass_best,lp_SFR_med,lp_sSFR_best,CFHT_u_MAG,HSC_g_MAG,HSC_r_MAG,HSC_i_MAG,HSC_z_MAG,HSC_y_MAG,UVISTA_J_MAG,UVISTA_H_MAG,UVISTA_Ks_MAG,IRAC_CH1_MAG,IRAC_CH2_MAG,u-g,g-r,r-i,i-z,z-y,y-J,J-H
7,12,150.476809,2.331511,0,0.432424,0.006043,0.017385,0.367684,0.004799,0.013929,0.539085,0.006673,0.012705,0.913139,0.006940,0.008187,0.995689,0.010382,0.011179,1.010389,0.021430,0.021652,1.176385,0.018240,0.016771,1.238664,0.023098,0.021265,1.273614,0.031158,0.026693,0.793401,0.012663,0.021113,0.623383,0.011264,0.024039,0.7070,0,NaN,30,0.00,5,9.048000e+08,0.1,1,-17.59872,-18.85507,-18.93651,8.69380,8.65194,8.74706,8.64422,-0.18369,-8.815,24.810225,24.986313,24.570856,23.998658,23.904690,23.888779,23.723626,23.667616,23.637405,24.151268,24.413112,-0.176089,0.415457,0.572198,0.093968,0.015911,0.165153,0.056011
39,73,149.468025,1.625609,0,1.057806,0.006321,0.007477,1.301316,0.007003,0.005769,2.870638,0.011023,0.003954,3.669017,0.012785,0.003763,4.321371,0.018494,0.004597,5.147967,0.045866,0.009110,5.366495,0.022263,0.004492,5.903986,0.031370,0.006063,6.666023,0.046399,0.007598,4.142945,0.036884,0.011779,4.426406,0.054762,0.016462,0.4290,0,NaN,30,0.34,9,1.609000e+09,0.2,1,-17.40718,-19.16920,-19.50800,9.06086,9.01925,9.10800,9.01447,0.08064,-8.977,23.838985,23.614043,22.755054,22.488626,22.310946,22.120911,22.075773,21.972137,21.840333,22.356727,22.284872,0.224942,0.858989,0.266428,0.177680,0.190035,0.045137,0.103637
40,74,150.488035,1.875422,0,0.941051,0.003713,0.004937,1.032136,0.004159,0.004320,1.066959,0.005443,0.005254,1.123350,0.005748,0.005525,1.208387,0.008979,0.007981,1.734654,0.019518,0.011504,2.413042,0.014805,0.006643,2.547439,0.017875,0.008007,2.257407,0.024673,0.011931,2.951406,0.013003,0.005829,3.054308,0.014661,0.006387,1.5087,0,NaN,29,2.23,10,9.048000e+08,0.1,1,-20.37010,-21.24365,-21.42674,9.49598,9.44923,9.54398,9.50023,0.99993,-8.536,23.965967,23.865658,23.829631,23.773713,23.694485,23.301968,22.943588,22.884741,23.015975,22.724928,22.687718,0.100310,0.036027,0.055918,0.079228,0.392517,0.358380,0.058847
46,85,150.476436,2.334178,0,0.685237,0.005227,0.009492,0.669388,0.004202,0.006701,0.744438,0.005884,0.008114,1.119968,0.005803,0.005582,1.484551,0.008772,0.006335,1.619744,0.018884,0.011903,1.782537,0.015855,0.009621,1.867551,0.020190,0.012329,2.274360,0.027033,0.012969,2.573668,0.012908,0.006634,1.992754,0.011691,0.007805,1.0800,0,NaN,29,2.19,5,2.273000e+08,0.2,0,-19.16843,-20.47925,-20.51206,9.14079,9.10236,9.18581,9.12799,0.90000,-8.234,24.310399,24.335805,24.220428,23.776986,23.471012,23.376384,23.272404,23.221819,23.007852,22.873619,23.151366,-0.025406,0.115376,0.443442,0.305974,0.094628,0.103980,0.050585
57,104,149.475722,1.626313,0,0.389645,0.004690,0.015066,0.377765,0.005186,0.014719,0.468260,0.008271,0.018192,0.582758,0.009130,0.016919,1.032440,0.013830,0.014389,1.231677,0.034203,0.028396,1.327044,0.017144,0.013989,1.671119,0.023782,0.016241,1.823997,0.035115,0.021015,2.302721,0.033234,0.019095,2.590386,0.044562,0.022890,1.2817,0,NaN,29,1.01,7,5.709000e+08,0.2,0,-18.94804,-20.47051,-20.73870,9.42859,9.36886,9.49659,9.40345,0.76785,-8.597,24.923327,24.956946,24.723783,24.486279,23.865338,23.673758,23.592787,23.342482,23.247440,22.994396,22.866589,-0.033620,0.233163,0.237504,0.620941,0.191580,0.080971,0.250305
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...

In [6]:
# RUN UMAP EMBEDDINGS FOR DIFFERENT RANDOM SEEDS W/ FIXED HYPERPARAMS
n_embeddings = 25

for i in range(n_embeddings):
    print(i)
    embedding_i = umap.UMAP(n_neighbors=70, min_dist=0.0, n_components=3, metric='manhattan', random_state=i, densmap=False).fit(data[['u-g', 'g-r', 'r-i', 'i-z', 'z-y', 'y-J', 'J-H']])
    data['UMAPrs'+str(i)+'-1'] = embedding_i.embedding_[:,0]
    data['UMAPrs'+str(i)+'-2'] = embedding_i.embedding_[:,1]
    data['UMAPrs'+str(i)+'-3'] = embedding_i.embedding_[:,2]

0


/Applications/anaconda3/envs/env1/lib/python3.11/site-packages/umap/umap_.py:1943: UserWarning: n_jobs value -1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(f"n_jobs value {self.n_jobs} overridden to 1 by setting random_state. Use no seed for parallelism.")


1


/Applications/anaconda3/envs/env1/lib/python3.11/site-packages/umap/umap_.py:1943: UserWarning: n_jobs value -1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(f"n_jobs value {self.n_jobs} overridden to 1 by setting random_state. Use no seed for parallelism.")


2


...

/Applications/anaconda3/envs/env1/lib/python3.11/site-packages/umap/umap_.py:1943: UserWarning: n_jobs value -1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(f"n_jobs value {self.n_jobs} overridden to 1 by setting random_state. Use no seed for parallelism.")


23


/Applications/anaconda3/envs/env1/lib/python3.11/site-packages/umap/umap_.py:1943: UserWarning: n_jobs value -1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(f"n_jobs value {self.n_jobs} overridden to 1 by setting random_state. Use no seed for parallelism.")


24


/Applications/anaconda3/envs/env1/lib/python3.11/site-packages/umap/umap_.py:1943: UserWarning: n_jobs value -1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(f"n_jobs value {self.n_jobs} overridden to 1 by setting random_state. Use no seed for parallelism.")


In [7]:
#data.to_parquet('umap_nn70_rs25_260523.parquet')

In [8]:
print('WOOHOOOOOOOOO!!!')

WOOHOOOOOOOOO!!!


In [12]:
data = pd.read_parquet('df7_maps_250819.parquet')
spec95 = pd.read_parquet('spec95_ugrizyJH_maps_250819.parquet')
data = data.drop(columns=['SOM-1', 'SOM-2', 'UMAP3D-1', 'UMAP3D-2', 'UMAP3D-3'])
spec95 = spec95.drop(columns=['SOM-1', 'SOM-2', 'UMAP3D-1', 'UMAP3D-2', 'UMAP3D-3'])
data = data[~data['ID'].isin(spec95[spec95['specz']<0.01]['ID'])]
spec95 = spec95[spec95['specz']>0.01]
data

,ID,RA,DEC,FLAG_COMBINED,CFHT_u_FLUX,CFHT_u_FLUXERR,CFHT_u_MAGERR,HSC_g_FLUX,HSC_g_FLUXERR,HSC_g_MAGERR,HSC_r_FLUX,HSC_r_FLUXERR,HSC_r_MAGERR,HSC_i_FLUX,HSC_i_FLUXERR,HSC_i_MAGERR,HSC_z_FLUX,HSC_z_FLUXERR,HSC_z_MAGERR,HSC_y_FLUX,HSC_y_FLUXERR,HSC_y_MAGERR,UVISTA_J_FLUX,UVISTA_J_FLUXERR,UVISTA_J_MAGERR,UVISTA_H_FLUX,UVISTA_H_FLUXERR,UVISTA_H_MAGERR,UVISTA_Ks_FLUX,UVISTA_Ks_FLUXERR,UVISTA_Ks_MAGERR,IRAC_CH1_FLUX,IRAC_CH1_FLUXERR,IRAC_CH1_MAGERR,IRAC_CH2_FLUX,IRAC_CH2_FLUXERR,IRAC_CH2_MAGERR,lp_photoz,lp_type,lp_zp_2,lp_NbFilt,lp_zq,lp_model,lp_age,lp_dust,lp_Attenuation,lp_MNUV,lp_MR,lp_MJ,lp_mass_med,lp_mass_med_min68,lp_mass_med_max68,lp_mass_best,lp_SFR_med,lp_sSFR_best,CFHT_u_MAG,HSC_g_MAG,HSC_r_MAG,HSC_i_MAG,HSC_z_MAG,HSC_y_MAG,UVISTA_J_MAG,UVISTA_H_MAG,UVISTA_Ks_MAG,IRAC_CH1_MAG,IRAC_CH2_MAG,u-g,g-r,r-i,i-z,z-y,y-J,J-H
7,12,150.476809,2.331511,0,0.432424,0.006043,0.017385,0.367684,0.004799,0.013929,0.539085,0.006673,0.012705,0.913139,0.006940,0.008187,0.995689,0.010382,0.011179,1.010389,0.021430,0.021652,1.176385,0.018240,0.016771,1.238664,0.023098,0.021265,1.273614,0.031158,0.026693,0.793401,0.012663,0.021113,0.623383,0.011264,0.024039,0.7070,0,NaN,30,0.00,5,9.048000e+08,0.1,1,-17.59872,-18.85507,-18.93651,8.69380,8.65194,8.74706,8.64422,-0.18369,-8.815,24.810225,24.986313,24.570856,23.998658,23.904690,23.888779,23.723626,23.667616,23.637405,24.151268,24.413112,-0.176089,0.415457,0.572198,0.093968,0.015911,0.165153,0.056011
39,73,149.468025,1.625609,0,1.057806,0.006321,0.007477,1.301316,0.007003,0.005769,2.870638,0.011023,0.003954,3.669017,0.012785,0.003763,4.321371,0.018494,0.004597,5.147967,0.045866,0.009110,5.366495,0.022263,0.004492,5.903986,0.031370,0.006063,6.666023,0.046399,0.007598,4.142945,0.036884,0.011779,4.426406,0.054762,0.016462,0.4290,0,NaN,30,0.34,9,1.609000e+09,0.2,1,-17.40718,-19.16920,-19.50800,9.06086,9.01925,9.10800,9.01447,0.08064,-8.977,23.838985,23.614043,22.755054,22.488626,22.310946,22.120911,22.075773,21.972137,21.840333,22.356727,22.284872,0.224942,0.858989,0.266428,0.177680,0.190035,0.045137,0.103637
40,74,150.488035,1.875422,0,0.941051,0.003713,0.004937,1.032136,0.004159,0.004320,1.066959,0.005443,0.005254,1.123350,0.005748,0.005525,1.208387,0.008979,0.007981,1.734654,0.019518,0.011504,2.413042,0.014805,0.006643,2.547439,0.017875,0.008007,2.257407,0.024673,0.011931,2.951406,0.013003,0.005829,3.054308,0.014661,0.006387,1.5087,0,NaN,29,2.23,10,9.048000e+08,0.1,1,-20.37010,-21.24365,-21.42674,9.49598,9.44923,9.54398,9.50023,0.99993,-8.536,23.965967,23.865658,23.829631,23.773713,23.694485,23.301968,22.943588,22.884741,23.015975,22.724928,22.687718,0.100310,0.036027,0.055918,0.079228,0.392517,0.358380,0.058847
46,85,150.476436,2.334178,0,0.685237,0.005227,0.009492,0.669388,0.004202,0.006701,0.744438,0.005884,0.008114,1.119968,0.005803,0.005582,1.484551,0.008772,0.006335,1.619744,0.018884,0.011903,1.782537,0.015855,0.009621,1.867551,0.020190,0.012329,2.274360,0.027033,0.012969,2.573668,0.012908,0.006634,1.992754,0.011691,0.007805,1.0800,0,NaN,29,2.19,5,2.273000e+08,0.2,0,-19.16843,-20.47925,-20.51206,9.14079,9.10236,9.18581,9.12799,0.90000,-8.234,24.310399,24.335805,24.220428,23.776986,23.471012,23.376384,23.272404,23.221819,23.007852,22.873619,23.151366,-0.025406,0.115376,0.443442,0.305974,0.094628,0.103980,0.050585
57,104,149.475722,1.626313,0,0.389645,0.004690,0.015066,0.377765,0.005186,0.014719,0.468260,0.008271,0.018192,0.582758,0.009130,0.016919,1.032440,0.013830,0.014389,1.231677,0.034203,0.028396,1.327044,0.017144,0.013989,1.671119,0.023782,0.016241,1.823997,0.035115,0.021015,2.302721,0.033234,0.019095,2.590386,0.044562,0.022890,1.2817,0,NaN,29,1.01,7,5.709000e+08,0.2,0,-18.94804,-20.47051,-20.73870,9.42859,9.36886,9.49659,9.40345,0.76785,-8.597,24.923327,24.956946,24.723783,24.486279,23.865338,23.673758,23.592787,23.342482,23.247440,22.994396,22.866589,-0.033620,0.233163,0.237504,0.620941,0.191580,0.080971,0.250305
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...

In [13]:
# loop over n_neighbors hyperparameter arguments (nn_hps) to generate multiple embeddings
nn_hps = [60, 70, 81]

for nn_hp in nn_hps:
    emb_i = umap.UMAP(n_neighbors=nn_hp, min_dist=0.0, metric='manhattan', n_components=3, random_state=41, densmap=False).fit(data[['u-g', 'g-r', 'r-i', 'i-z', 'z-y', 'y-J', 'J-H']])
    data['UMAPnn'+str(nn_hp)+'-1'] = emb_i.embedding_[:,0]
    data['UMAPnn'+str(nn_hp)+'-2'] = emb_i.embedding_[:,1]
    data['UMAPnn'+str(nn_hp)+'-3'] = emb_i.embedding_[:,2]

/Applications/anaconda3/envs/env1/lib/python3.11/site-packages/umap/umap_.py:1943: UserWarning: n_jobs value -1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(f"n_jobs value {self.n_jobs} overridden to 1 by setting random_state. Use no seed for parallelism.")
/Applications/anaconda3/envs/env1/lib/python3.11/site-packages/umap/umap_.py:1943: UserWarning: n_jobs value -1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(f"n_jobs value {self.n_jobs} overridden to 1 by setting random_state. Use no seed for parallelism.")
/Applications/anaconda3/envs/env1/lib/python3.11/site-packages/umap/umap_.py:1943: UserWarning: n_jobs value -1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(f"n_jobs value {self.n_jobs} overridden to 1 by setting random_state. Use no seed for parallelism.")


In [14]:
data.to_parquet('umap_nn607081_250819.parquet')

In [15]:
print('WOOOHOOOOO!!!!')

WOOOHOOOOO!!!!


In [16]:
data

,ID,RA,DEC,FLAG_COMBINED,CFHT_u_FLUX,CFHT_u_FLUXERR,CFHT_u_MAGERR,HSC_g_FLUX,HSC_g_FLUXERR,HSC_g_MAGERR,HSC_r_FLUX,HSC_r_FLUXERR,HSC_r_MAGERR,HSC_i_FLUX,HSC_i_FLUXERR,HSC_i_MAGERR,HSC_z_FLUX,HSC_z_FLUXERR,HSC_z_MAGERR,HSC_y_FLUX,HSC_y_FLUXERR,HSC_y_MAGERR,UVISTA_J_FLUX,UVISTA_J_FLUXERR,UVISTA_J_MAGERR,UVISTA_H_FLUX,UVISTA_H_FLUXERR,UVISTA_H_MAGERR,UVISTA_Ks_FLUX,UVISTA_Ks_FLUXERR,UVISTA_Ks_MAGERR,IRAC_CH1_FLUX,IRAC_CH1_FLUXERR,IRAC_CH1_MAGERR,IRAC_CH2_FLUX,IRAC_CH2_FLUXERR,IRAC_CH2_MAGERR,lp_photoz,lp_type,lp_zp_2,lp_NbFilt,lp_zq,lp_model,lp_age,lp_dust,lp_Attenuation,lp_MNUV,lp_MR,lp_MJ,lp_mass_med,lp_mass_med_min68,lp_mass_med_max68,lp_mass_best,lp_SFR_med,lp_sSFR_best,CFHT_u_MAG,HSC_g_MAG,HSC_r_MAG,HSC_i_MAG,HSC_z_MAG,HSC_y_MAG,UVISTA_J_MAG,UVISTA_H_MAG,UVISTA_Ks_MAG,IRAC_CH1_MAG,IRAC_CH2_MAG,u-g,g-r,r-i,i-z,z-y,y-J,J-H,UMAPnn60-1,UMAPnn60-2,UMAPnn60-3,UMAPnn70-1,UMAPnn70-2,UMAPnn70-3,UMAPnn81-1,UMAPnn81-2,UMAPnn81-3
7,12,150.476809,2.331511,0,0.432424,0.006043,0.017385,0.367684,0.004799,0.013929,0.539085,0.006673,0.012705,0.913139,0.006940,0.008187,0.995689,0.010382,0.011179,1.010389,0.021430,0.021652,1.176385,0.018240,0.016771,1.238664,0.023098,0.021265,1.273614,0.031158,0.026693,0.793401,0.012663,0.021113,0.623383,0.011264,0.024039,0.7070,0,NaN,30,0.00,5,9.048000e+08,0.1,1,-17.59872,-18.85507,-18.93651,8.69380,8.65194,8.74706,8.64422,-0.18369,-8.815,24.810225,24.986313,24.570856,23.998658,23.904690,23.888779,23.723626,23.667616,23.637405,24.151268,24.413112,-0.176089,0.415457,0.572198,0.093968,0.015911,0.165153,0.056011,-0.881290,3.796339,-1.209693,-0.741683,3.692303,-0.950502,-0.697438,3.672165,-0.903362
39,73,149.468025,1.625609,0,1.057806,0.006321,0.007477,1.301316,0.007003,0.005769,2.870638,0.011023,0.003954,3.669017,0.012785,0.003763,4.321371,0.018494,0.004597,5.147967,0.045866,0.009110,5.366495,0.022263,0.004492,5.903986,0.031370,0.006063,6.666023,0.046399,0.007598,4.142945,0.036884,0.011779,4.426406,0.054762,0.016462,0.4290,0,NaN,30,0.34,9,1.609000e+09,0.2,1,-17.40718,-19.16920,-19.50800,9.06086,9.01925,9.10800,9.01447,0.08064,-8.977,23.838985,23.614043,22.755054,22.488626,22.310946,22.120911,22.075773,21.972137,21.840333,22.356727,22.284872,0.224942,0.858989,0.266428,0.177680,0.190035,0.045137,0.103637,0.871471,6.029686,-1.179094,0.968271,5.938120,-1.001814,1.027934,5.904968,-0.902367
40,74,150.488035,1.875422,0,0.941051,0.003713,0.004937,1.032136,0.004159,0.004320,1.066959,0.005443,0.005254,1.123350,0.005748,0.005525,1.208387,0.008979,0.007981,1.734654,0.019518,0.011504,2.413042,0.014805,0.006643,2.547439,0.017875,0.008007,2.257407,0.024673,0.011931,2.951406,0.013003,0.005829,3.054308,0.014661,0.006387,1.5087,0,NaN,29,2.23,10,9.048000e+08,0.1,1,-20.37010,-21.24365,-21.42674,9.49598,9.44923,9.54398,9.50023,0.99993,-8.536,23.965967,23.865658,23.829631,23.773713,23.694485,23.301968,22.943588,22.884741,23.015975,22.724928,22.687718,0.100310,0.036027,0.055918,0.079228,0.392517,0.358380,0.058847,0.533604,4.124071,10.108575,0.640105,4.060228,10.181063,0.581099,4.021331,10.299862
46,85,150.476436,2.334178,0,0.685237,0.005227,0.009492,0.669388,0.004202,0.006701,0.744438,0.005884,0.008114,1.119968,0.005803,0.005582,1.484551,0.008772,0.006335,1.619744,0.018884,0.011903,1.782537,0.015855,0.009621,1.867551,0.020190,0.012329,2.274360,0.027033,0.012969,2.573668,0.012908,0.006634,1.992754,0.011691,0.007805,1.0800,0,NaN,29,2.19,5,2.273000e+08,0.2,0,-19.16843,-20.47925,-20.51206,9.14079,9.10236,9.18581,9.12799,0.90000,-8.234,24.310399,24.335805,24.220428,23.776986,23.471012,23.376384,23.272404,23.221819,23.007852,22.873619,23.151366,-0.025406,0.115376,0.443442,0.305974,0.094628,0.103980,0.050585,-1.864844,-0.585670,5.083677,-1.836826,-0.611205,5.174289,-1.713699,-0.614399,5.192638
57,104,149.475722,1.626313,0,0.389645,0.004690,0.015066,0.377765,0.005186,0.014719,0.468260,0.008271,0.018192,0.582758,0.009130,0.016919,1.032440,0.013830,0.014389,1.231677,0.034203,0.028396,1.327044,0.017144,0.013989,1.671119,0.023782,0.016241,1.823997